In [ ]:


import os

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
#путь в оригинале был кагловский
MODEL_PATH = "rubert-tiny2-finetuned-full/best"
MAX_SEQ_LENGTH = 128
OUTPUT_DIR = "embeddings"

TRAIN_PATH = "train.parquet"
BENCH_QUERIES_PATH = "benchmark_queries.parquet"
BENCH_ITEMS_PATH = "benchmark_items.parquet"


def build_item_text(row) -> str: # можно было и красивее, см. readme
    parts = [row.get("item_title_raw"), row.get("item_description_raw")]
    return " ".join(p for p in parts if isinstance(p, str) and p)


def build_query_text(row) -> str:
    return row.get("search_query") or ""


def save_embeddings(prefix: str, ids: list, embeddings: np.ndarray) -> None:
    np.save(f"{OUTPUT_DIR}/{prefix}_ids.npy", np.array(ids, dtype=object))
    np.save(f"{OUTPUT_DIR}/{prefix}_emb.npy", embeddings.astype("float32"))
    print(f"saved {prefix}")


os.makedirs(OUTPUT_DIR, exist_ok=True)

model = SentenceTransformer(MODEL_PATH)
model.max_seq_length = MAX_SEQ_LENGTH

train = pd.read_parquet(TRAIN_PATH)
train["query_text"] = train["search_query"].fillna("").str.strip().str.lower()

train_items = train.drop_duplicates(subset="item_id").reset_index(drop=True)
train_item_ids = train_items["item_id"].tolist()
train_item_texts = train_items.apply(build_item_text, axis=1).tolist()
emb = model.encode(train_item_texts, batch_size=256, convert_to_tensor=True,
                    normalize_embeddings=True, show_progress_bar=True).cpu().numpy()
save_embeddings("train_items", train_item_ids, emb)

train_query_texts = sorted(train["query_text"].unique().tolist())
emb = model.encode(train_query_texts, batch_size=256, convert_to_tensor=True,
                    normalize_embeddings=True, show_progress_bar=True).cpu().numpy()
save_embeddings("train_queries", train_query_texts, emb)

bench_items = pd.read_parquet(BENCH_ITEMS_PATH)
bench_item_ids = bench_items["item_id"].tolist()
bench_item_texts = bench_items.apply(build_item_text, axis=1).tolist()
emb = model.encode(bench_item_texts, batch_size=256, convert_to_tensor=True,
                    normalize_embeddings=True, show_progress_bar=True).cpu().numpy()
save_embeddings("bench_items", bench_item_ids, emb)

bench_queries = pd.read_parquet(BENCH_QUERIES_PATH)
bench_query_ids = bench_queries["query_id"].tolist()
bench_query_texts = bench_queries.apply(build_query_text, axis=1).tolist()
emb = model.encode(bench_query_texts, batch_size=256, convert_to_tensor=True,
                    normalize_embeddings=True, show_progress_bar=True).cpu().numpy()
save_embeddings("bench_queries", bench_query_ids, emb)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Batches:   0%|          | 0/1347 [00:00<?, ?it/s]

saved train_items


Batches:   0%|          | 0/292 [00:00<?, ?it/s]

saved train_queries


Batches:   0%|          | 0/740 [00:00<?, ?it/s]

saved bench_items


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

saved bench_queries


In [ ]:
import gc
import random
import re
import time
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# --- пути ---
TRAIN_PATH = "train.parquet"
BENCH_QUERIES_PATH = "benchmark_queries.parquet"
BENCH_ITEMS_PATH = "/benchmark_items.parquet"
EMBEDDINGS_DIR = "embeddings"
ANSWER_PATH = "answer.csv"
MODEL_SAVE_PATH = "mlp_scorer_full.pt"


EMB_DIM = 312
SEED = 42
VAL_FRAC = 0.15 
TOP_K = 50
TRAIN_QKEYS = 354463
N_NEG_DENSE = 40    #разные виды негативов 
N_NEG_LOC = 30     
N_NEG_RANDOM = 30   
VAL_EVAL_QKEYS = 3000

BATCH_SIZE = 1024
N_EPOCHS = 10
LR = 1e-5
HIDDEN = 256
MICROCAT_DIM = 16
Q_CHUNK, I_CHUNK = 8, 32_768

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"

torch.manual_seed(SEED)

_VID_RE = re.compile(r"Вид услуги\s*(.*?)(?=\s+Тип услуги|\s+Место оказания услуг|\s+Тип стоимости|$)")
_TIP_RE = re.compile(r"Тип услуги(?!\s+автосервиса)\s*(.*?)(?=\s+Место оказания услуг|\s+Тип стоимости|$)")


def parse_service_type(text): #получаем фичу про тип услуги
    if not isinstance(text, str) or not text.strip():
        return None, None
    v, t = _VID_RE.search(text), _TIP_RE.search(text)
    return (v.group(1).strip() if v else None), (t.group(1).strip() if t else None)


# Словари категорий -> коды
LOC, VID, TIP = {}, {}, {}


def code_of(vocab, v):
    if v is None or (isinstance(v, float) and v != v):
        return -1
    return vocab.setdefault(v, len(vocab))


def codes(vocab, values):
    return torch.tensor([code_of(vocab, v) for v in values], device=DEVICE)


def service_codes(infms):
    parsed = [parse_service_type(t) for t in infms]
    return codes(VID, [p[0] for p in parsed]), codes(TIP, [p[1] for p in parsed])

@dataclass
class QSide:
    emb: torch.Tensor
    loc: torch.Tensor
    delivery: torch.Tensor
    vid: torch.Tensor
    tip: torch.Tensor

@dataclass
class ISide:
    emb: torch.Tensor
    loc: torch.Tensor
    vid: torch.Tensor
    tip: torch.Tensor
    static: torch.Tensor    # [log price, rating, log reviews]
    microcat: torch.Tensor

def load_embeddings(prefix):
    ids = np.load(f"{EMBEDDINGS_DIR}/{prefix}_ids.npy", allow_pickle=True).tolist()
    emb = np.load(f"{EMBEDDINGS_DIR}/{prefix}_emb.npy")
    return ids, torch.from_numpy(emb).to(DEVICE)

def build_query_side(emb, df):
    """df — таблица запросов с колонками search_location_id / search_is_delivery_search / search_infm_params_text."""
    delivery = pd.to_numeric(df["search_is_delivery_search"], errors="coerce").fillna(0).to_numpy("float32")
    vid, tip = service_codes(df["search_infm_params_text"].tolist())
    return QSide(
        emb=emb.to(DEVICE),
        loc=codes(LOC, df["search_location_id"].tolist()),
        delivery=torch.tensor(delivery, device=DEVICE),
        vid=vid,
        tip=tip,
    )

ITEM_COLS = [
    "item_location_id",
    "item_price",
    "item_rating",
    "item_rating_reviews_count",
    "item_microcat_id",
    "item_infm_params_text",
]

def build_item_side(items_df, item_ids, emb, microcat_vocab): #делаем итемы
    df = items_df.drop_duplicates("item_id").set_index("item_id")[ITEM_COLS].reindex(item_ids)

    price = np.log1p(pd.to_numeric(df["item_price"], errors="coerce").clip(lower=0))
    rating = pd.to_numeric(df["item_rating"], errors="coerce")
    reviews = np.log1p(pd.to_numeric(df["item_rating_reviews_count"], errors="coerce").clip(lower=0))

    static = np.stack([
        s.fillna(s.median()).to_numpy("float32")
        for s in [price, rating, reviews]
    ], axis=1)

    # неизвестная / пропущенная микрокатегория 
    oov = len(microcat_vocab)
    vid, tip = service_codes(df["item_infm_params_text"].tolist())

    return ISide(
        emb=emb,
        loc=codes(LOC, df["item_location_id"].tolist()),
        vid=vid,
        tip=tip,
        static=torch.tensor(np.nan_to_num(static), device=DEVICE),
        microcat=torch.tensor([microcat_vocab.get(v, oov) for v in df["item_microcat_id"]], device=DEVICE),
    )


def match(a, b):
    m = (a == b).float()
    return torch.where((a < 0) | (b < 0), torch.full_like(m, -1.0), m)


def pair_inputs(Q, I, qi, ii):
    """Входы модели для пар (запрос qi, айтем ii)."""
    q_emb, i_emb = Q.emb[qi], I.emb[ii]
    numeric = torch.stack([
        (q_emb * i_emb).sum(-1),       # cosine
        match(Q.loc[qi], I.loc[ii]),   # location
        Q.delivery[qi],                # запрос с доставкой
        I.static[ii, 0],               # log price
        I.static[ii, 1],               # rating
        I.static[ii, 2],               # log reviews
        match(Q.vid[qi], I.vid[ii]),   # вид услуги
        match(Q.tip[qi], I.tip[ii]),   # тип услуги
    ], dim=-1)
    return q_emb, i_emb, numeric, I.microcat[ii]

N_NUMERIC = 8

class ScorerMLP(nn.Module):
    def __init__(self, num_microcats):
        super().__init__()
        self.microcat_emb = nn.Embedding(num_microcats + 1, MICROCAT_DIM)
        self.register_buffer("num_mean", torch.zeros(N_NUMERIC))
        self.register_buffer("num_std", torch.ones(N_NUMERIC))
        self.net = nn.Sequential(
            nn.Linear(EMB_DIM * 4 + N_NUMERIC + MICROCAT_DIM, HIDDEN),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(HIDDEN, HIDDEN // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(HIDDEN // 2, 1),
        )

    def forward(self, q, i, numeric, mc):
        numeric = (numeric - self.num_mean) / self.num_std
        x = torch.cat([q, i, q * i, (q - i).abs(), numeric, self.microcat_emb(mc)], dim=-1)
        return self.net(x).squeeze(-1)

def forward_rows(model, Q, I, qi, ii):
    q, i, numeric, mc = pair_inputs(Q, I, qi, ii)
    return model(q, i, numeric, mc)

def pairwise_loss(pos_scores, neg_scores):
    """Позитив должен получать скор выше негатива."""
    return -F.logsigmoid(pos_scores - neg_scores).mean()

def dense_topk(q_emb, i_emb, k, q_chunk=256):
    out = []
    for s in range(0, len(q_emb), q_chunk):
        out.append((q_emb[s:s + q_chunk] @ i_emb.T).topk(k, dim=1).indices.cpu())
    return torch.cat(out).numpy()


def dense_topk_in_groups(q_emb, q_group, group_positions, i_emb, k): #топ k но по группе
    by_group, res = defaultdict(list), {}

    for qi, g in enumerate(q_group):
        if g >= 0 and g in group_positions:
            by_group[g].append(qi)

    for g, qis in by_group.items():
        pos = group_positions[g]
        sub = i_emb[torch.as_tensor(pos, device=DEVICE)]
        top = dense_topk(q_emb[qis], sub, min(k, len(pos)))
        for row, qi in enumerate(qis):
            res[qi] = pos[top[row]]

    return res


@torch.no_grad()
def topk_all_items(model, Q, I, q_rows, top_k=TOP_K): #полный топ k
    
    model.eval()
    n_items = I.emb.shape[0]
    out = []
    t0 = time.time()

    for s in range(0, len(q_rows), Q_CHUNK):
        qs = q_rows[s:s + Q_CHUNK]

        # текущий топ-k для чанка запросов
        best_v = torch.full((len(qs), top_k), -1e9, device=DEVICE)
        best_i = torch.zeros((len(qs), top_k), dtype=torch.long, device=DEVICE)

        for a in range(0, n_items, I_CHUNK):
            items = torch.arange(a, min(a + I_CHUNK, n_items), device=DEVICE)
            qi = qs.repeat_interleave(len(items))
            ii = items.repeat(len(qs))

            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                logits = forward_rows(model, Q, I, qi, ii)

            logits = logits.float().view(len(qs), len(items))

            # склеиваем со старым топом и снова берём top_k
            cat_v = torch.cat([best_v, logits], dim=1)
            cat_i = torch.cat([best_i, items.unsqueeze(0).expand(len(qs), -1)], dim=1)
            best_v, idx = cat_v.topk(top_k, dim=1)
            best_i = cat_i.gather(1, idx)

        out.append(best_i.cpu())

    return torch.cat(out).numpy()


def recall_at_k(top, pos_lists, k=TOP_K):
    vals = [
        np.isin(pos, row[:k]).mean()   # доля позитивов, попавших в топ-k
        for row, pos in zip(top, pos_lists)
        if len(pos)
    ]
    return float(np.mean(vals)) if vals else 0.0

In [3]:
train = pd.read_parquet(TRAIN_PATH)
train["query_text"] = train["search_query"].fillna("").str.strip().str.lower()

search_cols = [c for c in train.columns if c.startswith("search_")]
train["qkey"] = train.groupby(search_cols, dropna=False, sort=False).ngroup()

texts = sorted(train["query_text"].unique().tolist())
random.Random(SEED).shuffle(texts)
val_texts = set(texts[:max(1, int(len(texts) * VAL_FRAC))])

qdf = train.drop_duplicates("qkey")[["qkey", "query_text"] + search_cols].set_index("qkey")
is_val = qdf["query_text"].isin(val_texts)

rng_py = random.Random(SEED)
train_sel = rng_py.sample(
    qdf.index[~is_val].tolist(),
    min(TRAIN_QKEYS, int((~is_val).sum()))
)
val_sel = rng_py.sample(
    qdf.index[is_val].tolist(),
    min(VAL_EVAL_QKEYS, int(is_val.sum()))
)

sel = train_sel + val_sel 
n_tr, n_val = len(train_sel), len(val_sel)

print(
    f"составных запросов всего: {len(qdf)}; "
    f"train берём {n_tr}, val для оценки {n_val}"
)

# релевантные айтемы для выбранных запросов
rel = (
    train[train["qkey"].isin(set(sel))]
    .groupby("qkey")["item_id"]
    .apply(set)
    .to_dict()
)
# уникальные айтемы и словарь микрокатегорий
items_df = train.drop_duplicates("item_id")[["item_id"] + ITEM_COLS]
microcat_vocab = {
    v: i
    for i, v in enumerate(sorted(items_df["item_microcat_id"].dropna().unique()))
}

del train
gc.collect()


train_item_ids, train_item_emb = load_embeddings("train_items")
item_id_to_idx = {iid: i for i, iid in enumerate(train_item_ids)}

q_text_ids, q_text_emb = load_embeddings("train_queries")
text_to_row = {t: i for i, t in enumerate(q_text_ids)}

qinfo = qdf.loc[sel]

QS = build_query_side(q_text_emb[[text_to_row[t] for t in qinfo["query_text"]]], qinfo)
I = build_item_side(items_df, train_item_ids, train_item_emb, microcat_vocab)

n_items = len(train_item_ids)

print(
    f"корпус train: {n_items} items; "
    f"без эмбеддинга: {len(set(items_df['item_id']) - set(item_id_to_idx))}"
)

# позитивы каждого запроса — индексы в корпусе эмбеддингов (в порядке sel)
pos_of = [
    np.array(
        sorted(item_id_to_idx[p] for p in rel[qk] if p in item_id_to_idx),
        dtype=np.int64,
    )
    for qk in sel
]


# группируем айтемы по локации
loc_np = I.loc.cpu().numpy()
valid = np.where(loc_np >= 0)[0]
order = valid[np.argsort(loc_np[valid], kind="stable")]

uniq, start = np.unique(loc_np[order], return_index=True)

loc_groups = {
    int(c): order[s:e]
    for c, s, e in zip(uniq, start, np.append(start[1:], len(order)))
}

# кандидаты в негативы; запас +10, т.к. часть окажется позитивами и будет отфильтрована
dense_top = dense_topk(QS.emb[:n_tr], I.emb, N_NEG_DENSE + 10)

loc_top = dense_topk_in_groups(
    QS.emb[:n_tr],
    QS.loc[:n_tr].tolist(),
    loc_groups,
    I.emb,
    N_NEG_LOC + 10,
)

rng = np.random.default_rng(SEED)


def first_n(cands, banned, n):
    return [p for p in cands if p not in banned][:n]


# Один позитив -  один негатив. Знаю, не очень хорошо, но время поджимало
qi_parts, ii_parts = [], []
pair_q_parts, pair_pos_parts, pair_neg_parts = [], [], []

for r in range(n_tr):
    pos = set(pos_of[r].tolist())
    if not pos:
        continue

    # если так подумать, то как то уже и смысла не имеет это все дело, но работает
    neg = first_n(dense_top[r].tolist(), pos, N_NEG_DENSE)
    neg += first_n(loc_top.get(r, np.empty(0, dtype=np.int64)).tolist(), pos, N_NEG_LOC)
    neg += first_n(rng.integers(0, n_items, N_NEG_RANDOM).tolist(), pos, N_NEG_RANDOM)

    neg = np.array(sorted(set(neg)), dtype=np.int64)

    if len(neg) == 0:
        continue

    qi_parts.append(np.full(len(pos_of[r]) + len(neg), r))
    ii_parts += [pos_of[r], neg]

    k = len(pos_of[r])
    pair_q_parts.append(np.full(k, r))
    pair_pos_parts.append(pos_of[r])
    pair_neg_parts.append(neg[np.arange(k) % len(neg)])

qi_t = torch.as_tensor(np.concatenate(qi_parts), device=DEVICE)
ii_t = torch.as_tensor(np.concatenate(ii_parts), device=DEVICE)

# тройки (query, positive, negative) — по ним считается pairwise-лосс
pair_q_t = torch.as_tensor(np.concatenate(pair_q_parts), dtype=torch.long, device=DEVICE)
pair_pos_t = torch.as_tensor(np.concatenate(pair_pos_parts), dtype=torch.long, device=DEVICE)
pair_neg_t = torch.as_tensor(np.concatenate(pair_neg_parts), dtype=torch.long, device=DEVICE)


model = ScorerMLP(len(microcat_vocab)).to(DEVICE)

sub = torch.randperm(len(qi_t), device=DEVICE)[:500_000]
numeric_sub = torch.cat([
    pair_inputs(QS, I, qi_t[sub[s:s + 100_000]], ii_t[sub[s:s + 100_000]])[2]
    for s in range(0, len(sub), 100_000)
])

model.num_mean.copy_(numeric_sub.mean(0))
model.num_std.copy_(numeric_sub.std(0) + 1e-6)

del numeric_sub, sub

val_rows = torch.arange(n_tr, n_tr + n_val, device=DEVICE)
val_pos = pos_of[n_tr:]

base_top = dense_topk(QS.emb[n_tr:], I.emb, TOP_K)
print(f"БАЗА (только dense cosine, все items): recall@50 = {recall_at_k(base_top, val_pos):.4f}")

# ОБУЧЕНИЕ
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_recall = -1.0

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    t0 = time.time()
    total_loss = 0.0
    n_batches = 0

    perm = torch.randperm(len(pair_q_t), device=DEVICE)

    for s in range(0, len(pair_q_t), BATCH_SIZE):
        b = perm[s:s + BATCH_SIZE]
        pos_scores = forward_rows(model, QS, I, pair_q_t[b], pair_pos_t[b])
        neg_scores = forward_rows(model, QS, I, pair_q_t[b], pair_neg_t[b])

        loss = pairwise_loss(pos_scores, neg_scores)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    top = topk_all_items(model, QS, I, val_rows)
    recall = recall_at_k(top, val_pos)

    print(
        f"epoch {epoch}: "
        f"loss={total_loss / n_batches:.4f} "
        f"recall@50={recall:.4f} "
        f"[{time.time() - t0:.0f}s]"
    )

    if recall > best_recall:
        best_recall = recall
        torch.save(
            {"model_state": model.state_dict(), "microcat_vocab": microcat_vocab},
            MODEL_SAVE_PATH,
        )
        print(f"  -> новый лучший чекпоинт: {MODEL_SAVE_PATH}")

print(f"Лучший recall@50 = {best_recall:.4f}")


составных запросов всего: 354463; train берём 304748, val для оценки 3000
корпус train: 344825 items; без эмбеддинга: 0
БАЗА (только dense cosine, все items): recall@50 = 0.2056
epoch 1: loss=0.6644 recall@50=0.7620 [218s]
  -> новый лучший чекпоинт: /kaggle/working/mlp_scorer_full.pt
epoch 2: loss=0.5360 recall@50=0.7905 [216s]
  -> новый лучший чекпоинт: /kaggle/working/mlp_scorer_full.pt
epoch 3: loss=0.3976 recall@50=0.8025 [216s]
  -> новый лучший чекпоинт: /kaggle/working/mlp_scorer_full.pt
epoch 4: loss=0.3218 recall@50=0.8034 [216s]
  -> новый лучший чекпоинт: /kaggle/working/mlp_scorer_full.pt
epoch 5: loss=0.2843 recall@50=0.7998 [216s]
epoch 6: loss=0.2656 recall@50=0.7984 [216s]
epoch 7: loss=0.2562 recall@50=0.7974 [216s]
epoch 8: loss=0.2506 recall@50=0.7955 [216s]
epoch 9: loss=0.2474 recall@50=0.7961 [216s]
epoch 10: loss=0.2438 recall@50=0.7962 [216s]
Лучший recall@50 = 0.8034


In [ ]:
# грузим лучший чекпоинт с диска
ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])

bench_queries = pd.read_parquet(BENCH_QUERIES_PATH)
bench_items = pd.read_parquet(BENCH_ITEMS_PATH)

bench_item_ids, bench_item_emb = load_embeddings("bench_items")
IB = build_item_side(bench_items, bench_item_ids, bench_item_emb, microcat_vocab)

raw_qids, raw_qemb = load_embeddings("bench_queries")
raw_row = {q: i for i, q in enumerate(raw_qids)}

bq_ids = bench_queries["query_id"].tolist()
QB = build_query_side(raw_qemb[[raw_row[q] for q in bq_ids]], bench_queries)

top = topk_all_items(model, QB, IB, torch.arange(len(bq_ids), device=DEVICE))

answer = pd.DataFrame({
    "query_id": bq_ids,
    "answer": [" ".join(bench_item_ids[p] for p in row) for row in top],
})

answer.to_csv(ANSWER_PATH, index=False)

print(
    f"готово: {ANSWER_PATH}, "
    f"строк={len(answer)}, "
    f"уникальных query_id={answer['query_id'].nunique()}"
)

готово: /kaggle/working/answer.csv, строк=2452, уникальных query_id=2452
